# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR^2) Exploration with `mlcroissant`

This notebook demonstrates how to explore and process the FAIR\^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All dataset entities such as record sets, fields, and columns are referenced by their `@id` values for clarity and reproducibility.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata and available records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n\nIdentifier: {metadata.identifier}\nLicense: {metadata.license}\nVersion: {metadata.version}")

## 2. Data Overview
Explore the available record sets and their fields using their `@id` identifiers.

In [ ]:
# List all Record Sets along with their @id and fields
record_sets_info = []
for rs in dataset.record_sets():
    rs_info = {
        '@id': rs['@id'],
        'name': rs.get('name', '(no name)'),
        'fields': []
    }
    fields = rs.get('field') or []
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        if isinstance(field, dict):
            rs_info['fields'].append({'@id': field.get('@id'), 'name': field.get('name', '')})
        else:
            rs_info['fields'].append({'@id': str(field), 'name': ''})
    record_sets_info.append(rs_info)

for rs in record_sets_info:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs['name']}")
    for field in rs['fields']:
        print(f"    Field @id: {field['@id']}, Name: {field['name']}")
    print('-'*50)

if not record_sets_info:
    print("No record sets found. Please check the dataset schema for available tabular resources.")

## 3. Data Extraction
Load data from the available record set(s) into pandas DataFrames for analysis. All record set and field references use their `@id` values.

In [ ]:
# Identify record sets to extract data from
record_set_ids = [rs['@id'] for rs in record_sets_info]
dataframes = {}

# Attempt to load each record set as a DataFrame using mlcroissant's records interface
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[record_set_id])} records from RecordSet @id: {record_set_id}")
        print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
    except Exception as e:
        print(f"Could not load records for RecordSet @id: {record_set_id}: {e}")

# Display the first few rows of the main record set (use the first if only one present)
main_rs_id = record_set_ids[0] if record_set_ids else None
if main_rs_id in dataframes:
    display(dataframes[main_rs_id].head())
else:
    print("No dataframes available to display.")

## 4. Exploratory Data Analysis (EDA)
Perform foundational EDA using one numeric field and one grouping/categorical field, referencing fields by their `@id`. This includes filtering records, normalization, and grouping.

In [ ]:
# For illustration, select likely numeric and group fields by examining column names
numeric_field_id = None
group_field_id = None
if main_rs_id in dataframes:
    df = dataframes[main_rs_id]
    # Try to find likely numeric and group fields by name/ID heuristics
    for col in df.columns:
        # Pick first column matching possible numeric indicators
        if numeric_field_id is None and any(substr in col.lower() for substr in ['age', 'interval', 'count', 'duration', 'years']):
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
        # Pick first column matching group indicators
        if group_field_id is None and any(substr in col.lower() for substr in ['sex', 'gender', 'group', 'site', 'anatom', 'location']):
            group_field_id = col
    
    if numeric_field_id is None:
        # fallback to first numerical column
        for col in df.select_dtypes(include='number').columns:
            numeric_field_id = col
            break
    if group_field_id is None:
        # fallback to first non-numeric column
        for col in df.select_dtypes(exclude='number').columns:
            group_field_id = col
            break

    print(f"Using numeric_field_id: {numeric_field_id}, group_field_id: {group_field_id}")

    threshold = 10
    if numeric_field_id is not None and numeric_field_id in df.columns:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where field '{numeric_field_id}' > {threshold} (n={len(filtered_df)}):")
        display(filtered_df.head())

        # Normalization
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a categorical field if present
        if group_field_id is not None and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
            display(grouped_df.head())
    else:
        print("No appropriate numeric field found for EDA.")
else:
    print("No main record set dataframe available for EDA.")

## 5. Visualization
Plot the distribution of the chosen numeric field and, if available, a bar plot grouped by the selected group field. These use only `@id` references of fields/columns.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id in dataframes and numeric_field_id is not None and numeric_field_id in dataframes[main_rs_id].columns:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[main_rs_id][numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    # Barplot by group
    if group_field_id is not None and group_field_id in dataframes[main_rs_id].columns:
        plt.figure(figsize=(10,4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=dataframes[main_rs_id], ci=None)
        plt.title(f"Mean '{numeric_field_id}' by '{group_field_id}'")
        plt.ylabel(f"Mean of {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.xticks(rotation=30)
        plt.show()
else:
    print("No suitable data available for visualization.")

## 6. Conclusion

- We loaded and explored the FAIR\^2 dataset using `mlcroissant`, referencing all dataset entities by their `@id`s for clarity and reproducibility.
- Available record sets and fields were inspected; data was loaded into DataFrames, and basic exploratory data analysis and visualization were performed using field IDs.
- Further analysis can leverage the structured metadata and clear identifiers provided by the Croissant schema for advanced, reproducible workflows.
